# CIFAR-100 Transfer Learning — ConvNeXt-Tiny (Bước 3: Chuẩn hóa Train / Val / Test + Seed + Early Stopping)

In [1]:
import os
import random
import time
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset

In [2]:
SEED = 42

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

seed_everything(SEED)

if torch.cuda.is_available():
    device = torch.device("cuda")
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True  # ưu tiên tốc độ trên L4
    props = torch.cuda.get_device_properties(0)
    print(f"Device   : cuda ({props.name})")
    print(f"VRAM     : {props.total_memory / 1e9:.2f} GB")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Device   : mps (Apple Silicon)")
else:
    device = torch.device("cpu")
    print("Device   : cpu")

print(f"CPU cores: {os.cpu_count()} | SEED: {SEED}")

Device   : cuda (NVIDIA L4)
VRAM     : 23.66 GB
CPU cores: 12 | SEED: 42


In [3]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: User cancelled dfs_ephemeral authorization

### Data Preprocess (`224×224`) & Stratified Split: Train (45k) / Val (5k) / Test (10k)

In [ ]:
data_path = "/content/drive/MyDrive/Cifar-100/data"

# Dùng MEAN/STD chuẩn ImageNet để khớp với trọng số pretrained của ConvNeXt-Tiny
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(
        224,
        scale=(0.8, 1.0),
        interpolation=transforms.InterpolationMode.BICUBIC,
    ),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.15)),
])

eval_transform = transforms.Compose([
    transforms.Resize(
        (224, 224),
        interpolation=transforms.InterpolationMode.BICUBIC,
    ),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

need_download = not os.path.exists(os.path.join(data_path, "cifar-100-python"))

# Tạo 2 bản CIFAR-100 train riêng biệt để Val KHÔNG bị dính augmentation của Train (tránh Transform Leakage)
full_train_aug = torchvision.datasets.CIFAR100(
    root=data_path, train=True, download=need_download, transform=train_transform
)
full_train_clean = torchvision.datasets.CIFAR100(
    root=data_path, train=True, download=False, transform=eval_transform
)

# Stratified Split: mỗi class lấy đúng 50 ảnh làm Val, 450 ảnh làm Train (100 classes -> 5,000 Val / 45,000 Train)
targets = np.array(full_train_aug.targets)
rng = np.random.default_rng(SEED)
train_idx, val_idx = [], []
val_per_class = 50

for cls in range(100):
    cls_indices = np.where(targets == cls)[0]
    rng.shuffle(cls_indices)
    val_idx.extend(cls_indices[:val_per_class])
    train_idx.extend(cls_indices[val_per_class:])

trainset = Subset(full_train_aug, train_idx)
valset = Subset(full_train_clean, val_idx)

# Hold-out Test set (10,000 ảnh): TUYỆT ĐỐI không dùng để chọn model trong lúc train
testset = torchvision.datasets.CIFAR100(
    root=data_path, train=False, download=need_download, transform=eval_transform
)

is_cuda = device.type == "cuda"
num_workers = min(6, os.cpu_count() or 2)
loader_generator = torch.Generator().manual_seed(SEED)

trainloader = DataLoader(
    trainset,
    batch_size=128,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=is_cuda,
    persistent_workers=num_workers > 0,
    prefetch_factor=4 if num_workers > 0 else None,
    worker_init_fn=seed_worker,
    generator=loader_generator,
    drop_last=True,
)

valloader = DataLoader(
    valset,
    batch_size=256,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=is_cuda,
    persistent_workers=num_workers > 0,
    prefetch_factor=4 if num_workers > 0 else None,
)

testloader = DataLoader(
    testset,
    batch_size=256,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=is_cuda,
    persistent_workers=num_workers > 0,
    prefetch_factor=4 if num_workers > 0 else None,
)

print(f"Train samples: {len(trainset):,} (450/class)")
print(f"Val   samples: {len(valset):,} (50/class)")
print(f"Test  samples: {len(testset):,} (100/class — chỉ chấm 1 lần cuối cùng)")

In [ ]:
images, labels = next(iter(trainloader))
print("Batch images shape:", images.shape)            # Kỳ vọng: torch.Size([128, 3, 224, 224])
print("Num classes       :", len(testset.classes))  # Kỳ vọng: 100

### Model: ConvNeXt-Tiny (Pretrained ImageNet-1K)

In [ ]:
seed_everything(SEED)  # Cố định seed trước khi khởi tạo lớp Linear mới

weights = models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1
model = models.convnext_tiny(weights=weights)

# Thay lớp phân loại cuối từ 1000 class (ImageNet) -> 100 class (CIFAR-100)
in_features = model.classifier[2].in_features
model.classifier[2] = nn.Linear(in_features, 100)

model = model.to(device)

### Sanity Check & Parameters

In [ ]:
dummy_input = torch.randn(4, 3, 224, 224, device=device)

model.eval()
with torch.no_grad():
    output = model(dummy_input)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Input shape         : {dummy_input.shape}")
print(f"Output shape        : {output.shape}")
print(f"Total parameters    : {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

### Loss, Optimizer (Differential LR), Scheduler & Evaluate

In [ ]:
epochs = 15
warmup_epochs = 2
early_stop_patience = 5  # Dừng sớm nếu Val Acc không cải thiện sau 5 epoch liên tiếp

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Differential LR: Backbone học chậm để giữ tri thức ImageNet, Head mới học nhanh gấp 10 lần
backbone_params = [p for n, p in model.named_parameters() if not n.startswith("classifier")]
head_params = [p for n, p in model.named_parameters() if n.startswith("classifier")]

optimizer = optim.AdamW(
    [
        {"params": backbone_params, "lr": 1e-4},
        {"params": head_params, "lr": 1e-3},
    ],
    weight_decay=1e-2,
)

warmup_scheduler = optim.lr_scheduler.LinearLR(
    optimizer, start_factor=0.1, end_factor=1.0, total_iters=warmup_epochs
)
cosine_scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=epochs - warmup_epochs, eta_min=1e-6
)
scheduler = optim.lr_scheduler.SequentialLR(
    optimizer, schedulers=[warmup_scheduler, cosine_scheduler], milestones=[warmup_epochs]
)

In [ ]:
def evaluate(model, dataloader, criterion):
    model.eval()
    eval_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device, non_blocking=is_cuda)
            labels = labels.to(device, non_blocking=is_cuda)
            with torch.amp.autocast("cuda", dtype=torch.bfloat16, enabled=is_cuda):
                outputs = model(images)
                loss = criterion(outputs, labels)

            eval_loss += loss.item()
            preds = torch.argmax(outputs, dim=1)
            total += labels.size(0)
            correct += (preds == labels).sum().item()

    accuracy = 100.0 * correct / total
    eval_loss = eval_loss / len(dataloader)
    return eval_loss, accuracy

### Fine-tune Model (Chọn Best Checkpoint trên `valloader` + Early Stopping)

In [ ]:
train_losses, train_accuracies = [], []
val_losses, val_accuracies = [], []
best_val_acc = 0.0
no_improve_epochs = 0
save_path = "/content/drive/MyDrive/Cifar-100/best_convnext_tiny_v6_split.pth"

for epoch in range(epochs):
    t0 = time.time()
    model.train()
    running_loss = 0.0
    running_correct = 0
    total = 0

    for images, labels in trainloader:
        images = images.to(device, non_blocking=is_cuda)
        labels = labels.to(device, non_blocking=is_cuda)

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", dtype=torch.bfloat16, enabled=is_cuda):
            outputs = model(images)
            loss = criterion(outputs, labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        running_loss += loss.item()
        preds = torch.argmax(outputs, dim=1)
        total += labels.size(0)
        running_correct += (preds == labels).sum().item()

    backbone_lr = optimizer.param_groups[0]["lr"]
    head_lr = optimizer.param_groups[1]["lr"]
    scheduler.step()

    epoch_accuracy = 100.0 * running_correct / total
    epoch_loss = running_loss / len(trainloader)
    val_loss, val_accuracy = evaluate(model, valloader, criterion)

    if val_accuracy > best_val_acc:
        best_val_acc = val_accuracy
        no_improve_epochs = 0
        torch.save(
            {
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "best_val_acc": best_val_acc,
                "seed": SEED,
            },
            save_path,
        )
    else:
        no_improve_epochs += 1

    elapsed = time.time() - t0
    print(
        f"Epoch [{epoch + 1:2d}/{epochs}] ({elapsed:4.1f}s) | "
        f"LR(bb/hd): {backbone_lr:.1e}/{head_lr:.1e} | "
        f"Train Loss: {epoch_loss:.4f}, Acc: {epoch_accuracy:.2f}% | "
        f"Val Loss: {val_loss:.4f}, Acc: {val_accuracy:.2f}% (Best Val: {best_val_acc:.2f}%)"
    )

    train_losses.append(epoch_loss)
    train_accuracies.append(epoch_accuracy)
    val_losses.append(val_loss)
    val_accuracies.append(val_accuracy)

    if no_improve_epochs >= early_stop_patience:
        print(f"Early stopping kích hoạt tại Epoch {epoch + 1} (không cải thiện sau {early_stop_patience} epochs).")
        break

### Plot Training Curves (Train vs Validation)

In [ ]:
def save_training_plot(train_losses, val_losses, train_accs, val_accs, filename):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(train_losses, label="Train Loss", linewidth=2)
    axes[0].plot(val_losses, label="Val Loss", linewidth=2)
    axes[0].set_title("Loss qua các Epoch (Train vs Val)", fontsize=12, fontweight="bold")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].legend()
    axes[0].grid(True, linestyle="--", alpha=0.6)

    best_acc_val = max(val_accs)
    best_epoch = val_accs.index(best_acc_val) + 1

    axes[1].plot(train_accs, label="Train Accuracy", linewidth=2)
    axes[1].plot(val_accs, label="Val Accuracy", linewidth=2)
    axes[1].axhline(
        best_acc_val,
        color="red",
        linestyle=":",
        alpha=0.7,
        label=f"Best Val: {best_acc_val:.2f}% (Ep {best_epoch})",
    )
    axes[1].set_title(f"Accuracy qua các Epoch (Best Val: {best_acc_val:.2f}%)", fontsize=12, fontweight="bold")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy (%)")
    axes[1].legend()
    axes[1].grid(True, linestyle="--", alpha=0.6)

    plt.tight_layout()
    plt.savefig(filename, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"Đã lưu biểu đồ thành công tại: {filename}")

save_training_plot(
    train_losses,
    val_losses,
    train_accuracies,
    val_accuracies,
    filename="/content/drive/MyDrive/Cifar-100/cifar100_v6_convnext_tiny_val_split.png",
)

### Final Evaluation trên Hold-out Test Set (Chấm ĐÚNG 1 lần duy nhất) + Error Analysis

In [ ]:
save_path = "/content/drive/MyDrive/Cifar-100/best_convnext_tiny_v6_split.pth"
checkpoint = torch.load(save_path, map_location=device, weights_only=True)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print(
    f"Đã load Best Checkpoint tại Epoch {checkpoint['epoch']} "
    f"| Best Val Acc: {checkpoint['best_val_acc']:.2f}% | Seed: {checkpoint['seed']}"
)

all_labels = []
all_preds = []
all_top5_correct = 0
all_confs = []

with torch.no_grad():
    for images, labels in testloader:
        images = images.to(device, non_blocking=is_cuda)
        labels = labels.to(device, non_blocking=is_cuda)

        with torch.amp.autocast("cuda", dtype=torch.bfloat16, enabled=is_cuda):
            outputs = model(images)

        probs = torch.softmax(outputs.float(), dim=1)
        confs, preds = torch.max(probs, dim=1)

        top5_preds = torch.topk(probs, k=5, dim=1).indices
        all_top5_correct += (top5_preds == labels.unsqueeze(1)).any(dim=1).sum().item()

        all_labels.append(labels.cpu())
        all_preds.append(preds.cpu())
        all_confs.append(confs.cpu())

all_labels = torch.cat(all_labels).numpy()
all_preds = torch.cat(all_preds).numpy()
all_confs = torch.cat(all_confs).numpy()

top1_acc = 100.0 * (all_preds == all_labels).mean()
top5_acc = 100.0 * all_top5_correct / len(all_labels)
num_errors = int((all_preds != all_labels).sum())

print(f"\n[HOLD-OUT TEST SET]")
print(f"Top-1 Accuracy : {top1_acc:.2f}% ({len(all_labels) - num_errors}/{len(all_labels)})")
print(f"Top-5 Accuracy : {top5_acc:.2f}%")
print(f"Tổng số ảnh sai: {num_errors}/{len(all_labels)}")

classes = testset.classes
per_class_acc = []
for cls_idx, cls_name in enumerate(classes):
    mask = all_labels == cls_idx
    acc = 100.0 * (all_preds[mask] == cls_idx).mean()
    per_class_acc.append((acc, cls_name, int((all_preds[mask] != cls_idx).sum()), int(mask.sum())))

per_class_acc.sort(key=lambda x: x[0])
print("\n--- Top 10 Class khó nhất ---")
for acc, name, errs, total_c in per_class_acc[:10]:
    print(f"  {name:18s}: {acc:5.1f}% (sai {errs:2d}/{total_c})")

wrong_pairs = Counter(
    (classes[t], classes[p])
    for t, p in zip(all_labels, all_preds)
    if t != p
)
print("\n--- Top 15 cặp Class hay bị nhầm nhất (Thật -> Dự đoán) ---")
for (true_c, pred_c), count in wrong_pairs.most_common(15):
    print(f"  {true_c:18s} -> {pred_c:18s}: {count:2d} lần")

In [ ]:
# Trực quan hóa 12 ảnh đoán SAI với độ tự tin cao nhất trên Hold-out Test Set
wrong_indices = np.where(all_preds != all_labels)[0]
sorted_wrong = wrong_indices[np.argsort(-all_confs[wrong_indices])]

fig, axes = plt.subplots(3, 4, figsize=(13, 9))
fig.suptitle("Top 12 ảnh đoán SAI tự tin nhất (ConvNeXt-Tiny v6 — Hold-out Test)", fontsize=14, fontweight="bold")

for ax, idx in zip(axes.flat, sorted_wrong[:12]):
    ax.imshow(testset.data[idx])
    true_name = classes[all_labels[idx]]
    pred_name = classes[all_preds[idx]]
    conf = all_confs[idx] * 100.0
    ax.set_title(f"Thật: {true_name}\nĐoán: {pred_name} ({conf:.1f}%)", fontsize=9, color="crimson")
    ax.axis("off")

plt.tight_layout()
mistake_plot_path = "/content/drive/MyDrive/Cifar-100/convnext_tiny_v6_top_mistakes.png"
plt.savefig(mistake_plot_path, dpi=200, bbox_inches="tight")
plt.show()
print(f"Đã lưu ảnh phân tích lỗi tại: {mistake_plot_path}")

### Test-Time Augmentation (TTA 2-view: Ảnh gốc + Lật ngang)

In [ ]:
def evaluate_tta_2view(model, dataloader):
    model.eval()
    correct_1, top5_1 = 0, 0
    correct_2, top5_2 = 0, 0
    total = 0

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device, non_blocking=is_cuda)
            labels = labels.to(device, non_blocking=is_cuda)
            images_flip = torch.flip(images, dims=[3])

            with torch.amp.autocast("cuda", dtype=torch.bfloat16, enabled=is_cuda):
                p_orig = torch.softmax(model(images).float(), dim=1)
                p_flip = torch.softmax(model(images_flip).float(), dim=1)

            probs_tta = 0.5 * (p_orig + p_flip)

            total += labels.size(0)
            correct_1 += (torch.argmax(p_orig, dim=1) == labels).sum().item()
            top5_1 += (torch.topk(p_orig, k=5, dim=1).indices == labels.unsqueeze(1)).any(dim=1).sum().item()

            correct_2 += (torch.argmax(probs_tta, dim=1) == labels).sum().item()
            top5_2 += (torch.topk(probs_tta, k=5, dim=1).indices == labels.unsqueeze(1)).any(dim=1).sum().item()

    acc1, t5_1 = 100.0 * correct_1 / total, 100.0 * top5_1 / total
    acc2, t5_2 = 100.0 * correct_2 / total, 100.0 * top5_2 / total

    print(f"Standard (1-view)         : Top-1 = {acc1:.2f}% | Top-5 = {t5_1:.2f}%")
    print(f"TTA 2-view (Orig + HFlip) : Top-1 = {acc2:.2f}% ({acc2 - acc1:+.2f}%) | Top-5 = {t5_2:.2f}%")

evaluate_tta_2view(model, testloader)